# AI Research Paper Q&A - Fine-Tuning

## QLoRA fine-tuning of Qwen3-4B-Instruct-2507 on QASPER (AI/ML research paper Q&A)

Run this on a free Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU).

Before running: upload `train.jsonl` and `validation.jsonl` (from your local
`data/` folder) into this Colab session's file browser (left sidebar -> folder icon -> upload).

In [ ]:
!pip install -q -U "transformers>=4.51.0,<5.0.0" "trl==0.19.1" peft accelerate bitsandbytes datasets huggingface_hub mlflow

In [ ]:
import os
from datetime import datetime

import torch
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import mlflow

In [ ]:
# Constants

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
PROJECT_NAME = "ai-research-qa"
HF_USER = "your-hf-username"  # <-- change this to your HuggingFace username

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/qwen3-4b-ai-research-qa-v2"

# Hyper-parameters - overall
# Tuned to use a free T4's full 16GB VRAM (a batch size of 1 was leaving most
# of it idle): bigger real batch size for smoother/less noisy gradients and
# faster wall-clock time, while grad accumulation keeps the effective batch
# size (4 x 4 = 16) large enough for stable training.

EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch size = 4 x 4 = 16
# If you hit a CUDA out-of-memory error when training starts, drop these two
# to PER_DEVICE_TRAIN_BATCH_SIZE=2 / GRADIENT_ACCUMULATION_STEPS=8 (keeps the
# same effective batch size of 16) and re-run from this cell down.
MAX_SEQUENCE_LENGTH = 512

# Hyper-parameters - QLoRA
# Targeting attention AND MLP layers (not just attention) is the biggest
# single lever for adapter quality - it lets LoRA reshape how the model
# processes information, not just how it attends. Paired with a higher
# rank (32) so the adapter has enough capacity to use that expanded target
# set well.

QUANT_4_BIT = True
LORA_R = 32
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.05

# Hyper-parameters - training

LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = "cosine"
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_8bit"  # 8-bit optimizer state to save T4 VRAM

# T4 (compute capability 7.5) does NOT support bf16 -> we use fp16
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

LOG_STEPS = 10
SAVE_STEPS = 100
LOG_TO_MLFLOW = True

set_seed(42)

In [ ]:
# On a free T4 this will print False -> confirms we correctly fall back to fp16 above
use_bf16

### Log in to HuggingFace

Click the key icon on the left sidebar in Colab -> add a secret named `HF_TOKEN` with your
HuggingFace token (needs write access to push the model).

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# MLflow tracking - logs loss curves, learning rate, and all training metrics.
# Using a local SQLite database inside the Colab session (the plain-folder
# './mlruns' backend is deprecated in newer MLflow); download mlflow.db at
# the end to keep it with your project.

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment(PROJECT_NAME)
os.environ["MLFLOW_EXPERIMENT_NAME"] = PROJECT_NAME
os.environ["HF_MLFLOW_LOG_ARTIFACTS"] = "true"

### Load the dataset

Upload `train.jsonl` and `validation.jsonl` into this Colab session first
(left sidebar -> folder icon -> upload button).

In [ ]:
dataset = load_dataset(
    "json",
    data_files={"train": "train.jsonl", "validation": "validation.jsonl"},
)
train = dataset["train"]
val = dataset["validation"]
print(f"Train examples: {len(train)}")
print(f"Validation examples: {len(val)}")

In [ ]:
# If a full run is too slow on free Colab, uncomment to reduce dataset size:
# train = train.select(range(3000))
# val = val.select(range(300))

## Load the Tokenizer and Model

The model is "quantized" to 4-bit (NF4) so it fits on a 16GB T4.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

# Set up the configuration for Training

Two objects: a `LoraConfig` for LoRA hyperparameters, and an `SFTConfig` for overall training.

In [ ]:
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="mlflow" if LOG_TO_MLFLOW else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

# Create the trainer

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
)

## Kick off fine-tuning

This will run for a while, saving + pushing a checkpoint to the Hub every `SAVE_STEPS` steps.
Free Colab can disconnect you without warning - if that happens, your last checkpoint is
already safe on the Hub. See the project README for how to resume.

In [ ]:
fine_tuning.train()

fine_tuning.model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Saved to the hub: {HUB_MODEL_NAME}")

In [ ]:
if LOG_TO_MLFLOW and mlflow.active_run():
    mlflow.end_run()

print("Download mlflow.db from the file browser on the left to keep your MLflow history.")